# Child-held-out comparison

This notebook does not train. It reads the ten-fold scores in `../main-output`, checks that the four models used one index list and one set of class counts, recomputes AUC from the saved softmax, and writes the tables and figures used in the chapter.

The scored unit is a new infant. Duplicate video ids were dropped before the split. The primary endpoint is validation accuracy. Nemenyi is the post-hoc decision. Wilcoxon is the fold-level paired test. McNemar is the clip-level paired test on the shared 1,404 predictions.


In [1]:
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import friedmanchisquare, wilcoxon
from sklearn.metrics import confusion_matrix, recall_score, roc_auc_score

plt.switch_backend("Agg")

ROOT = Path("..").resolve()
RESULT_DIR = ROOT / "main-output"
FIG_DIR = Path("figures")
OUT_DIR = Path("outputs")
FIG_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ["NomNomNom", "Peekaboo", "TearingPaper", "ThatsNotACat", "ThatsNotAHat"]
DEDUP_COUNTS = [280, 273, 279, 292, 280]
MODEL_NAMES = ["Multi-cue", "3D_CNN", "TAN", "CNN_LSTM"]
DISPLAY = {"Multi-cue": "Multi-cue", "3D_CNN": "3D CNN", "TAN": "TAN", "CNN_LSTM": "CNN-LSTM"}
PALETTE = {"Multi-cue": "#0072B2", "3D_CNN": "#D55E00", "TAN": "#009E73", "CNN_LSTM": "#CC79A7"}
NEMENYI_Q = {2: 1.960, 3: 2.343, 4: 2.569, 5: 2.728, 6: 2.850, 7: 2.949, 8: 3.031}
N_SPLITS = 10
BOOTSTRAP_DRAWS = 5000
BOOTSTRAP_SEED = 2025
N_CLIPS = 1404

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 12,
    "axes.linewidth": 0.8,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "legend.frameon": False,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.dpi": 300,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})
print("Paths:", RESULT_DIR, FIG_DIR.resolve(), OUT_DIR.resolve())


Paths: /Users/jeremiah/Documents/PHD/phd_thesis_joke-Publication -14-oct-complet/main-output /Users/jeremiah/Documents/PHD/phd_thesis_joke-Publication -14-oct-complet/analysis/figures /Users/jeremiah/Documents/PHD/phd_thesis_joke-Publication -14-oct-complet/analysis/outputs


In [2]:
def holm_stepdown(p_values, alpha=0.05):
    p_values = np.asarray(p_values, dtype=float)
    order = np.argsort(p_values)
    adjusted = np.empty(len(p_values))
    reject = np.zeros(len(p_values), dtype=bool)
    running = 0.0
    stopped = False
    for rank, idx in enumerate(order):
        running = max(running, (len(p_values) - rank) * float(p_values[idx]))
        adjusted[idx] = min(1.0, running)
        if stopped or adjusted[idx] > alpha:
            stopped = True
        else:
            reject[idx] = True
    return adjusted, reject


def load_fold(model_name, fold_id):
    path = RESULT_DIR / model_name.lower() / f"fold_{fold_id}_predictions.npz"
    if not path.exists():
        raise RuntimeError(f"Missing {path}")
    return np.load(path)


def load_predictions(model_name):
    true_parts, pred_parts, proba_parts, index_parts = [], [], [], []
    for fold_id in range(1, N_SPLITS + 1):
        saved = load_fold(model_name, fold_id)
        true_parts.append(saved["y_true"])
        pred_parts.append(saved["y_pred"])
        proba_parts.append(saved["y_proba"])
        index_parts.append(saved["val_idx"])
    return (
        np.concatenate(true_parts),
        np.concatenate(pred_parts),
        np.concatenate(proba_parts),
        np.concatenate(index_parts),
    )


def mcnemar_exact(b, c):
    """Exact two-sided McNemar on discordant counts b, c (A correct B wrong, A wrong B correct)."""
    n = int(b + c)
    if n == 0:
        return 1.0
    from math import comb
    k = min(b, c)
    # two-sided exact binomial under p=0.5
    p = sum(comb(n, i) for i in range(0, k + 1)) / (2 ** n)
    return float(min(1.0, 2.0 * p))


scores = pd.read_csv(RESULT_DIR / "all_models_performance.csv")
if set(scores["model"]) != set(MODEL_NAMES):
    raise RuntimeError(f"Expected {MODEL_NAMES}, found {sorted(scores['model'].unique())}.")
if int(scores.groupby("model").size().min()) != N_SPLITS:
    raise RuntimeError("Each model needs ten folds.")

frozen = None
frozen_path = RESULT_DIR / "shared_fold_indices.json"
if frozen_path.exists():
    import json
    frozen = json.loads(frozen_path.read_text())

problems = []
matrices = {}
aligned = {}  # model -> arrays ordered by clip index 0..1403

for model_name in MODEL_NAMES:
    y_true, y_pred, y_proba, val_idx = load_predictions(model_name)
    if len(val_idx) != N_CLIPS or not np.array_equal(np.sort(val_idx), np.arange(N_CLIPS)):
        problems.append(f"{model_name} does not cover each clip once.")
    counts = np.bincount(y_true, minlength=5).astype(int).tolist()
    if counts != DEDUP_COUNTS:
        problems.append(f"{model_name} class counts {counts} != {DEDUP_COUNTS}.")
    if not np.allclose(y_proba.sum(axis=1), 1.0, atol=1e-4):
        problems.append(f"{model_name} softmax rows do not sum to one.")
    if not np.array_equal(y_pred, y_proba.argmax(axis=1)):
        problems.append(f"{model_name} predictions are not the softmax argmax.")

    model_scores = scores[scores["model"] == model_name].set_index("fold")
    for fold_id in range(1, N_SPLITS + 1):
        saved = load_fold(model_name, fold_id)
        accuracy = float((saved["y_pred"] == saved["y_true"]).mean())
        auc = float(roc_auc_score(saved["y_true"], saved["y_proba"], multi_class="ovr", average="macro"))
        recorded = model_scores.loc[fold_id]
        if abs(accuracy - float(recorded["validation_accuracy"])) > 1e-8:
            problems.append(f"{model_name} fold {fold_id} accuracy does not match the CSV.")
        if abs(auc - float(recorded["auc_roc"])) > 1e-6:
            problems.append(f"{model_name} fold {fold_id} AUC does not match the CSV.")
        if int(recorded["n_correct"]) != int((saved["y_pred"] == saved["y_true"]).sum()):
            problems.append(f"{model_name} fold {fold_id} n_correct does not match.")
        if int(recorded["epochs_ran"]) != int(recorded["best_epoch"]):
            problems.append(f"{model_name} fold {fold_id} epochs_ran is not the selected epoch.")
        if frozen is not None:
            frozen_fold = frozen["folds"][fold_id - 1]
            if not np.array_equal(saved["val_idx"], np.asarray(frozen_fold["val_idx"])):
                problems.append(f"{model_name} fold {fold_id} val_idx differs from shared_fold_indices.json.")

    # Align predictions onto clip index order for McNemar / hard-clip analysis.
    order = np.argsort(val_idx)
    aligned[model_name] = {
        "y_true": y_true[order],
        "y_pred": y_pred[order],
        "y_proba": y_proba[order],
        "val_idx": val_idx[order],
        "correct": (y_pred[order] == y_true[order]),
    }
    matrices[model_name] = confusion_matrix(y_true, y_pred, labels=list(range(5)))
    if matrices[model_name].sum(axis=1).astype(int).tolist() != DEDUP_COUNTS:
        problems.append(f"{model_name} confusion-matrix row sums differ from the modelling counts.")

for model_name in MODEL_NAMES[1:]:
    if not np.array_equal(aligned["Multi-cue"]["val_idx"], aligned[model_name]["val_idx"]):
        problems.append(f"{model_name} aligned indices differ from Multi-cue.")
    if not np.array_equal(aligned["Multi-cue"]["y_true"], aligned[model_name]["y_true"]):
        problems.append(f"{model_name} labels differ from Multi-cue after alignment.")

for fold_id in range(1, N_SPLITS + 1):
    left = load_fold("Multi-cue", fold_id)["val_idx"]
    for model_name in MODEL_NAMES[1:]:
        right = load_fold(model_name, fold_id)["val_idx"]
        if not np.array_equal(left, right):
            problems.append(f"Fold {fold_id} indices differ for {model_name}.")

if problems:
    raise RuntimeError("Refusing the paired analysis.\n" + "\n".join(problems))
print("Shared indices, softmax audit, and class counts confirmed for all four models.")


Shared indices, softmax audit, and class counts confirmed for all four models.


In [3]:
wide = scores.pivot(index="fold", columns="model", values="validation_accuracy").loc[:, MODEL_NAMES]
rng = np.random.default_rng(BOOTSTRAP_SEED)
ci_rows = []
for model_name in MODEL_NAMES:
    values = wide[model_name].to_numpy()
    draws = np.mean(values[rng.integers(0, len(values), size=(BOOTSTRAP_DRAWS, len(values)))], axis=1)
    lo, hi = np.percentile(draws, [2.5, 97.5])
    ci_rows.append({
        "model": model_name,
        "mean_validation_accuracy": float(values.mean()),
        "sd": float(values.std(ddof=1)),
        "ci_low": float(lo),
        "ci_high": float(hi),
    })
ci_table = pd.DataFrame(ci_rows)
pooled = scores.groupby("model").apply(
    lambda frame: frame["n_correct"].sum() / frame["n_validation"].sum(), include_groups=False
)
summary = ci_table.set_index("model")
summary["pooled_accuracy"] = pooled
summary["mean_auc"] = scores.groupby("model")["auc_roc"].mean()
summary["epochs_min"] = scores.groupby("model")["epochs_ran"].min()
summary["epochs_max"] = scores.groupby("model")["epochs_ran"].max()
summary["epochs_mean"] = scores.groupby("model")["epochs_ran"].mean()
summary = summary.loc[MODEL_NAMES]
summary.to_csv(OUT_DIR / "summary.csv")
print(summary.rename(index=DISPLAY).round(4).to_string())

means = wide.mean()
sds = wide.std(ddof=1)
fig, ax = plt.subplots(figsize=(5.4, 3.6))
ax.bar(
    range(len(MODEL_NAMES)), [means[name] for name in MODEL_NAMES],
    yerr=[sds[name] for name in MODEL_NAMES],
    color=[PALETTE[name] for name in MODEL_NAMES], capsize=4, ecolor="0.15", width=0.72,
)
ax.set_xticks(range(len(MODEL_NAMES)))
ax.set_xticklabels([DISPLAY[name] for name in MODEL_NAMES])
ax.set_ylabel("Validation accuracy")
floor = float(means.min() - sds.max() - 0.04)
ax.set_ylim(max(0.70, floor), 1.0)
fig.tight_layout()
fig.savefig(FIG_DIR / "validation_accuracy.png", bbox_inches="tight")
fig.savefig(FIG_DIR / "validation_accuracy.pdf", bbox_inches="tight")
plt.close(fig)

# Fold-size context for the chapter
fold_sizes = scores[scores["model"] == "Multi-cue"][["fold", "n_validation", "n_validation_children"]].copy()
fold_sizes.to_csv(OUT_DIR / "fold_margins.csv", index=False)
print(fold_sizes.to_string(index=False))


           mean_validation_accuracy      sd  ci_low  ci_high  pooled_accuracy  mean_auc  epochs_min  epochs_max  epochs_mean
model                                                                                                                       
Multi-cue                    0.9676  0.0111  0.9610   0.9737           0.9672    0.9981          28          60         40.2
3D CNN                       0.9505  0.0333  0.9291   0.9684           0.9509    0.9988          67         113         89.0
TAN                          0.8909  0.0247  0.8761   0.9051           0.8910    0.9874          16          61         30.4
CNN-LSTM                     0.8263  0.0303  0.8083   0.8437           0.8262    0.9672          36          83         54.0


 fold  n_validation  n_validation_children
    1           142                      9
    2           133                      8
    3           146                      9
    4           162                     10
    5           141                      9
    6           153                      9
    7           141                      9
    8           124                      8
    9           140                      9
   10           122                      8


In [4]:
stat, friedman_p = friedmanchisquare(*[wide[name].to_numpy() for name in MODEL_NAMES])
ranks = wide.rank(axis=1, ascending=False, method="average")
mean_ranks = ranks.mean(axis=0).loc[MODEL_NAMES]
k = len(MODEL_NAMES)
critical_difference = float(NEMENYI_Q[k] * np.sqrt(k * (k + 1) / (6.0 * N_SPLITS)))
print(f"Friedman chi^2={stat:.3f}, p={friedman_p:.6g}")
print(f"Nemenyi critical difference={critical_difference:.3f}")
print(mean_ranks.rename(index=DISPLAY).sort_values().to_string())

pair_rows = []
for left, right in combinations(MODEL_NAMES, 2):
    gap = abs(float(mean_ranks[left] - mean_ranks[right]))
    pair_rows.append({
        "model_a": left,
        "model_b": right,
        "mean_rank_a": float(mean_ranks[left]),
        "mean_rank_b": float(mean_ranks[right]),
        "rank_gap": gap,
        "critical_difference": critical_difference,
        "nemenyi_significant": bool(gap > critical_difference),
    })
nemenyi = pd.DataFrame(pair_rows).sort_values("rank_gap", ascending=False)
nemenyi.to_csv(OUT_DIR / "nemenyi_pairwise.csv", index=False)
print(nemenyi.replace({"model_a": DISPLAY, "model_b": DISPLAY}).to_string(index=False))

ordered = list(mean_ranks.sort_values().index)
intervals = []
for start in range(len(ordered)):
    for end in range(start, len(ordered)):
        gap = abs(float(mean_ranks[ordered[end]] - mean_ranks[ordered[start]]))
        if gap <= critical_difference + 1e-12:
            intervals.append((start, end))
maximal = []
for start, end in intervals:
    covered = any(
        other_start <= start and other_end >= end and (other_start, other_end) != (start, end)
        for other_start, other_end in intervals
    )
    if not covered and end > start:
        maximal.append((start, end))

fig, ax = plt.subplots(figsize=(6.4, 2.4))
positions = {name: float(mean_ranks[name]) for name in ordered}
for name in ordered:
    ax.plot(positions[name], 0.0, "o", color=PALETTE[name], ms=8, zorder=3)
    ax.text(positions[name], -0.18, DISPLAY[name], ha="center", va="top", fontsize=10)
left = min(positions.values()) - 0.15
right = max(positions.values()) + 0.15
ax.plot([left, right], [0, 0], color="0.4", lw=1, zorder=1)
for level, (start, end) in enumerate(maximal):
    y = 0.12 + 0.16 * level
    ax.plot(
        [positions[ordered[start]], positions[ordered[end]]],
        [y, y], color="0.2", lw=2, solid_capstyle="round",
    )
ax.set_xlim(left, right)
ax.set_ylim(-0.55, 0.2 + 0.18 * max(len(maximal), 1))
ax.set_xlabel("Average rank")
ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_visible(False)
fig.tight_layout()
fig.savefig(FIG_DIR / "critical_difference.png", bbox_inches="tight")
fig.savefig(FIG_DIR / "critical_difference.pdf", bbox_inches="tight")
plt.close(fig)


Friedman chi^2=24.840, p=1.66767e-05
Nemenyi critical difference=1.483
model
Multi-cue    1.3
3D CNN       1.8
TAN          3.0
CNN-LSTM     3.9
  model_a  model_b  mean_rank_a  mean_rank_b  rank_gap  critical_difference  nemenyi_significant
Multi-cue CNN-LSTM          1.3          3.9       2.6             1.483213                 True
   3D CNN CNN-LSTM          1.8          3.9       2.1             1.483213                 True
Multi-cue      TAN          1.3          3.0       1.7             1.483213                 True
   3D CNN      TAN          1.8          3.0       1.2             1.483213                False
      TAN CNN-LSTM          3.0          3.9       0.9             1.483213                False
Multi-cue   3D CNN          1.3          1.8       0.5             1.483213                False


In [5]:
comparisons, raw_p = [], []
for left, right in combinations(MODEL_NAMES, 2):
    difference = (wide[left] - wide[right]).to_numpy()
    if np.allclose(difference, 0):
        statistic, p_value = np.nan, 1.0
    else:
        statistic, p_value = wilcoxon(wide[left], wide[right], zero_method="wilcox", alternative="two-sided")
        statistic, p_value = float(statistic), float(p_value)
    # Paired Cohen's d_z on fold differences
    sd = float(difference.std(ddof=1))
    dz = float(difference.mean() / sd) if sd > 0 else np.nan
    draws = np.mean(difference[rng.integers(0, len(difference), size=(BOOTSTRAP_DRAWS, len(difference)))], axis=1)
    lo, hi = np.percentile(draws, [2.5, 97.5])
    comparisons.append({
        "model_a": left,
        "model_b": right,
        "mean_difference": float(difference.mean()),
        "ci_low": float(lo),
        "ci_high": float(hi),
        "cohen_dz": dz,
        "wins": int((difference > 0).sum()),
        "losses": int((difference < 0).sum()),
        "ties": int((difference == 0).sum()),
        "wilcoxon_statistic": statistic,
        "p_value": p_value,
    })
    raw_p.append(p_value)
adjusted, reject = holm_stepdown(raw_p)
for row, adj, flag in zip(comparisons, adjusted, reject):
    row["p_holm"] = float(adj)
    row["holm_reject"] = bool(flag)
wilcoxon_table = pd.DataFrame(comparisons).sort_values("mean_difference", ascending=False)
wilcoxon_table.to_csv(OUT_DIR / "paired_accuracy_wilcoxon.csv", index=False)
print(wilcoxon_table.replace({"model_a": DISPLAY, "model_b": DISPLAY}).round(4).to_string(index=False))

fig, ax = plt.subplots(figsize=(6.2, 3.4))
shown = wilcoxon_table.reset_index(drop=True)
ypos = np.arange(len(shown))[::-1]
ax.axvline(0, color="0.45", lw=0.8)
ax.errorbar(
    shown["mean_difference"], ypos,
    xerr=[shown["mean_difference"] - shown["ci_low"], shown["ci_high"] - shown["mean_difference"]],
    fmt="o", color="0.15", ms=5, capsize=3, lw=1,
)
ax.set_yticks(ypos)
ax.set_yticklabels([f"{DISPLAY[a]} − {DISPLAY[b]}" for a, b in zip(shown["model_a"], shown["model_b"])])
ax.set_xlabel("Accuracy difference")
fig.tight_layout()
fig.savefig(FIG_DIR / "paired_accuracy_differences.png", bbox_inches="tight")
fig.savefig(FIG_DIR / "paired_accuracy_differences.pdf", bbox_inches="tight")
plt.close(fig)


  model_a  model_b  mean_difference  ci_low  ci_high  cohen_dz  wins  losses  ties  wilcoxon_statistic  p_value  p_holm  holm_reject
Multi-cue CNN-LSTM           0.1413  0.1186   0.1645    3.6552    10       0     0                 0.0   0.0020  0.0117         True
   3D CNN CNN-LSTM           0.1242  0.0910   0.1519    2.4245    10       0     0                 0.0   0.0020  0.0117         True
Multi-cue      TAN           0.0767  0.0631   0.0902    3.3198    10       0     0                 0.0   0.0020  0.0117         True
      TAN CNN-LSTM           0.0646  0.0418   0.0856    1.7106     9       1     0                 1.0   0.0039  0.0117         True
   3D CNN      TAN           0.0596  0.0335   0.0854    1.3181     9       1     0                 2.0   0.0059  0.0117         True
Multi-cue   3D CNN           0.0171 -0.0005   0.0383    0.5188     7       3     0                13.0   0.1602  0.1602        False


In [6]:
# Clip-level McNemar on the shared 1,404 predictions (same children, same clips).
mcnemar_rows = []
mc_p = []
for left, right in combinations(MODEL_NAMES, 2):
    a_ok = aligned[left]["correct"]
    b_ok = aligned[right]["correct"]
    both = int((a_ok & b_ok).sum())
    a_only = int((a_ok & ~b_ok).sum())
    b_only = int((~a_ok & b_ok).sum())
    neither = int((~a_ok & ~b_ok).sum())
    p_value = mcnemar_exact(a_only, b_only)
    mcnemar_rows.append({
        "model_a": left,
        "model_b": right,
        "both_correct": both,
        "a_only": a_only,
        "b_only": b_only,
        "both_wrong": neither,
        "discordant": a_only + b_only,
        "p_value": p_value,
    })
    mc_p.append(p_value)
mc_adj, mc_rej = holm_stepdown(mc_p)
for row, adj, flag in zip(mcnemar_rows, mc_adj, mc_rej):
    row["p_holm"] = float(adj)
    row["holm_reject"] = bool(flag)
mcnemar_table = pd.DataFrame(mcnemar_rows).sort_values("p_value")
mcnemar_table.to_csv(OUT_DIR / "mcnemar_clip_level.csv", index=False)
print(mcnemar_table.replace({"model_a": DISPLAY, "model_b": DISPLAY}).to_string(index=False))

# Hard clips: missed by every model, or by the top two.
y_true = aligned["Multi-cue"]["y_true"]
correct_stack = np.column_stack([aligned[name]["correct"] for name in MODEL_NAMES])
n_correct_models = correct_stack.sum(axis=1)
hard = pd.DataFrame({
    "clip_index": np.arange(N_CLIPS),
    "true_class": [CLASS_NAMES[i] for i in y_true],
    "n_models_correct": n_correct_models,
    "multi_cue_ok": correct_stack[:, 0],
    "cnn3d_ok": correct_stack[:, 1],
    "tan_ok": correct_stack[:, 2],
    "cnn_lstm_ok": correct_stack[:, 3],
})
hard["missed_by_all"] = hard["n_models_correct"] == 0
hard["missed_by_top_two"] = (~hard["multi_cue_ok"]) & (~hard["cnn3d_ok"])
hard["solved_only_by_multi_cue"] = (
    hard["multi_cue_ok"] & ~hard["cnn3d_ok"] & ~hard["tan_ok"] & ~hard["cnn_lstm_ok"]
)
hard_summary = (
    hard.groupby("true_class")
    .agg(
        n_clips=("clip_index", "size"),
        missed_by_all=("missed_by_all", "sum"),
        missed_by_top_two=("missed_by_top_two", "sum"),
        solved_only_by_multi_cue=("solved_only_by_multi_cue", "sum"),
    )
    .reindex(CLASS_NAMES)
)
hard_summary.to_csv(OUT_DIR / "hard_clip_summary.csv")
hard.to_csv(OUT_DIR / "hard_clips.csv", index=False)
print(hard_summary.to_string())
print(f"Clips missed by all four models: {int((n_correct_models == 0).sum())}")
print(f"Clips missed by Multi-cue and 3D CNN: {int((~aligned['Multi-cue']['correct'] & ~aligned['3D_CNN']['correct']).sum())}")

# Agreement heatmap of correct/incorrect patterns
agree = np.zeros((4, 4), dtype=float)
for i, a in enumerate(MODEL_NAMES):
    for j, b in enumerate(MODEL_NAMES):
        agree[i, j] = float((aligned[a]["correct"] == aligned[b]["correct"]).mean())
fig, ax = plt.subplots(figsize=(4.8, 4.2))
sns.heatmap(
    agree, annot=True, fmt=".3f", cmap="Greens", vmin=0.7, vmax=1.0,
    xticklabels=[DISPLAY[n] for n in MODEL_NAMES],
    yticklabels=[DISPLAY[n] for n in MODEL_NAMES],
    square=True, ax=ax, cbar_kws={"label": "Agreement on correctness"},
)
ax.set_title("Clip-level correctness agreement")
fig.tight_layout()
fig.savefig(FIG_DIR / "correctness_agreement.png", bbox_inches="tight")
fig.savefig(FIG_DIR / "correctness_agreement.pdf", bbox_inches="tight")
plt.close(fig)


  model_a  model_b  both_correct  a_only  b_only  both_wrong  discordant      p_value       p_holm  holm_reject
Multi-cue CNN-LSTM          1125     233      35          11         268 4.538852e-37 2.723311e-36         True
   3D CNN CNN-LSTM          1106     229      54          15         283 8.105995e-27 4.052997e-26         True
Multi-cue      TAN          1214     144      37           9         181 4.154180e-16 1.661672e-15         True
   3D CNN      TAN          1199     136      52          17         188 7.435700e-10 2.230710e-09         True
      TAN CNN-LSTM          1036     215     124          29         339 8.816592e-07 1.763318e-06         True
Multi-cue   3D CNN          1295      63      40           6         103 2.968221e-02 2.968221e-02         True
              n_clips  missed_by_all  missed_by_top_two  solved_only_by_multi_cue
true_class                                                                       
NomNomNom         280              0                

In [7]:
recall_rows = []
for model_name in MODEL_NAMES:
    y_true = aligned[model_name]["y_true"]
    y_pred = aligned[model_name]["y_pred"]
    recall = recall_score(y_true, y_pred, labels=list(range(5)), average=None, zero_division=0)
    for class_name, value, support in zip(CLASS_NAMES, recall, DEDUP_COUNTS):
        recall_rows.append({"model": model_name, "class": class_name, "recall": float(value), "support": support})
recall_long = pd.DataFrame(recall_rows)
recall_table = recall_long.pivot(index="class", columns="model", values="recall").loc[CLASS_NAMES, MODEL_NAMES]
recall_table.to_csv(OUT_DIR / "per_class_recall.csv")
print(recall_table.rename(columns=DISPLAY).round(3).to_string())

fig, ax = plt.subplots(figsize=(7.2, 3.8))
positions = np.arange(len(CLASS_NAMES))
bar_width = 0.18
for offset, model_name in enumerate(MODEL_NAMES):
    ax.bar(
        positions + (offset - 1.5) * bar_width,
        recall_table[model_name].to_numpy(),
        width=bar_width, color=PALETTE[model_name], label=DISPLAY[model_name],
    )
ax.set_xticks(positions)
ax.set_xticklabels(CLASS_NAMES, rotation=25, ha="right")
ax.set_ylabel("Recall")
ax.set_ylim(0, 1)
ax.legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, 1.16))
fig.tight_layout()
fig.savefig(FIG_DIR / "per_class_recall.png", bbox_inches="tight")
fig.savefig(FIG_DIR / "per_class_recall.pdf", bbox_inches="tight")
plt.close(fig)

# Timing descriptive table
timing = scores.groupby("model").agg(
    mean_ms=("inference_ms_per_clip", "mean"),
    sd_ms=("inference_ms_per_clip", "std"),
).loc[MODEL_NAMES]
timing.to_csv(OUT_DIR / "per_clip_time.csv")
print(timing.rename(index=DISPLAY).round(4).to_string())


model         Multi-cue  3D CNN    TAN  CNN-LSTM
class                                           
NomNomNom         0.957   0.964  0.939     0.836
Peekaboo          0.993   0.960  0.930     0.861
TearingPaper      0.953   0.896  0.789     0.814
ThatsNotACat      0.966   0.962  0.914     0.832
ThatsNotAHat      0.968   0.971  0.882     0.789


           mean_ms   sd_ms
model                     
Multi-cue   0.0055  0.0010
3D CNN      0.0922  0.0132
TAN         0.0073  0.0011
CNN-LSTM    0.0172  0.0016


In [8]:
def draw_matrix(cm, axis, title, vmax, show_y=True):
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues", vmin=0, vmax=vmax,
        square=True, cbar=False, ax=axis,
        xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES if show_y else False,
        linewidths=0.5, linecolor="white",
        annot_kws={"size": 11, "weight": "medium"},
    )
    for spine in axis.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.5)
    axis.set_title(title, pad=8)
    axis.set_xlabel("Predicted class")
    axis.set_ylabel("True class" if show_y else "")
    axis.tick_params(axis="x", labelrotation=35)
    for label in axis.get_xticklabels():
        label.set_ha("right")
    axis.tick_params(axis="y", labelrotation=0)


shared_max = max(int(cm.max()) for cm in matrices.values())
for model_name in MODEL_NAMES:
    fig, axis = plt.subplots(figsize=(6.6, 5.6))
    sns.heatmap(
        matrices[model_name], annot=True, fmt="d", cmap="Blues", vmin=0, vmax=shared_max,
        square=True, cbar=True, ax=axis,
        xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
        linewidths=0.6, linecolor="white",
        annot_kws={"size": 16, "weight": "medium"},
        cbar_kws={"shrink": 0.84, "label": "Clips"},
    )
    for spine in axis.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.6)
    axis.set_title(DISPLAY[model_name])
    axis.set_xlabel("Predicted class")
    axis.set_ylabel("True class")
    axis.tick_params(axis="x", labelrotation=30)
    for label in axis.get_xticklabels():
        label.set_ha("right")
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"{model_name.lower()}_confusion_matrix.png", bbox_inches="tight")
    fig.savefig(FIG_DIR / f"{model_name.lower()}_confusion_matrix.pdf", bbox_inches="tight")
    fig.savefig(RESULT_DIR / model_name.lower() / f"{model_name}_confusion_matrix.png", bbox_inches="tight")
    plt.close(fig)

fig, axes = plt.subplots(2, 2, figsize=(11.4, 10.4))
for index, (axis, model_name) in enumerate(zip(axes.ravel(), MODEL_NAMES)):
    draw_matrix(matrices[model_name], axis, DISPLAY[model_name], shared_max, show_y=(index % 2 == 0))
    if index < 2:
        axis.set_xlabel("")
fig.tight_layout(rect=[0, 0, 0.90, 1], h_pad=2.8, w_pad=1.8)
bar = fig.colorbar(
    plt.cm.ScalarMappable(norm=plt.Normalize(vmin=0, vmax=shared_max), cmap="Blues"),
    cax=fig.add_axes([0.92, 0.18, 0.018, 0.64]),
)
bar.set_label("Clips")
fig.savefig(FIG_DIR / "confusion_matrices.png", bbox_inches="tight")
fig.savefig(FIG_DIR / "confusion_matrices.pdf", bbox_inches="tight")
plt.close(fig)
print(f"Confusion matrices use one color scale, maximum count {shared_max}.")
for model_name, cm in matrices.items():
    print(DISPLAY[model_name], cm.sum(axis=1).astype(int).tolist())


Confusion matrices use one color scale, maximum count 282.
Multi-cue [280, 273, 279, 292, 280]
3D CNN [280, 273, 279, 292, 280]
TAN [280, 273, 279, 292, 280]
CNN-LSTM [280, 273, 279, 292, 280]


In [9]:
order = list(means.sort_values(ascending=False).index)
best, second = order[0], order[1]
mc_vs_3d = wilcoxon_table[(wilcoxon_table.model_a == "Multi-cue") & (wilcoxon_table.model_b == "3D_CNN")].iloc[0]
mcnemar_top = mcnemar_table[(mcnemar_table.model_a == "Multi-cue") & (mcnemar_table.model_b == "3D_CNN")].iloc[0]
missed_all = int((n_correct_models == 0).sum())
lines = [
    f"Best mean validation accuracy: {DISPLAY[best]} {means[best]:.3f}.",
    f"Second: {DISPLAY[second]} {means[second]:.3f}.",
    f"Friedman chi^2={stat:.3f}, p={friedman_p:.6g}. Critical difference={critical_difference:.3f}.",
    "Nemenyi is the post-hoc rank decision. Wilcoxon is the fold-level paired test. McNemar is the clip-level paired test.",
    (
        f"Multi-cue vs 3D CNN: fold mean Δ={mc_vs_3d['mean_difference']:+.3f}, "
        f"Holm p={mc_vs_3d['p_holm']:.3f}, Cohen d_z={mc_vs_3d['cohen_dz']:.2f}; "
        f"McNemar Multi-cue-only/3D-only {int(mcnemar_top['a_only'])}/{int(mcnemar_top['b_only'])}, "
        f"Holm p={mcnemar_top['p_holm']:.4g}."
    ),
    f"Clips missed by every model: {missed_all} / {N_CLIPS}.",
]
if order[:2] != ["Multi-cue", "3D_CNN"]:
    lines.append(
        f"The mean order is {' > '.join(DISPLAY[name] for name in order)}. "
        "It is not Multi-cue followed by 3D CNN."
    )
else:
    lines.append("The mean order is Multi-cue, then 3D CNN, then the remaining models by their means.")
(OUT_DIR / "reading.txt").write_text("\n".join(lines) + "\n")
print("\n".join(lines))


Best mean validation accuracy: Multi-cue 0.968.
Second: 3D CNN 0.950.
Friedman chi^2=24.840, p=1.66767e-05. Critical difference=1.483.
Nemenyi is the post-hoc rank decision. Wilcoxon is the fold-level paired test. McNemar is the clip-level paired test.
Multi-cue vs 3D CNN: fold mean Δ=+0.017, Holm p=0.160, Cohen d_z=0.52; McNemar Multi-cue-only/3D-only 63/40, Holm p=0.02968.
Clips missed by every model: 0 / 1404.
The mean order is Multi-cue, then 3D CNN, then the remaining models by their means.
